In [2]:
# --- MÓDULO DE INGESTÃO: EMPREGO SETORIAL (SERVIÇOS) ---

import pandas as pd

# Defino o ponto de entrada para os dados de vínculos formais do setor terciário.
# Esta variável é a "prova de fogo" do Efeito Multiplicador (Spillover): 
# a riqueza direta da mineração está se convertendo em empregos indiretos na cidade?
caminho_arquivo = r'C:\Users\fabio\TCC\9_Emprego_servicos.csv'

try:
    # Realizo a ingestão do dataset do setor de serviços.
    # skiprows=2: Salto o cabeçalho institucional duplo padrão das extrações da RAIS.
    # encoding='latin-1': Decodificação mandatória para plataformas governamentais legadas.
    df_emprego_servicos = pd.read_csv(
        caminho_arquivo,
        sep=';',
        skiprows=2,
        encoding='latin-1'
    )
    
    # DIAGNÓSTICO ESTRUTURAL INICIAL:
    # Confirmo a integridade da carga, validando o alinhamento das colunas após o salto (skiprows).
    print("--- Visualização do Dataset Setorial (Emprego no Setor Terciário) ---")
    print(df_emprego_servicos.head())
    
    print("\n--- Inventário de Séries Temporais (Setor de Serviços) ---")
    print(df_emprego_servicos.columns.tolist())

except Exception as e:
    print(f"ERRO DE INGESTÃO: Falha crítica ao carregar a matriz de Emprego em Serviços: {e}")

--- TABELA CARREGADA ---
                  Município  2021  2020  2019  2018  2017  2016  2015  2014  \
0  RO-ALTA FLORESTA D OESTE   354   328   374   361   367   434   316   324   
1              RO-ARIQUEMES  4668  4490  4730  4725  4498  4329  4547  4359   
2                 RO-CABIXI    46    46    51    51    56    49    39    33   
3                 RO-CACOAL  5815  5484  5626  5352  5196  4929  4988  4614   
4             RO-CEREJEIRAS   536   463   451   384   351   351   345   346   

   2013  ...  2011  2010  2009  2008  2007  2006  2005  2004  2003  2002  
0   301  ...   203   186   210   169   147   171   134   122   100   111  
1  3982  ...  3475  3214  3000  2754  2525  2264  2127  1939  1720  1460  
2    35  ...    28    26    18    13    13    12    12    10    10     8  
3  4307  ...  3525  3171  2873  2383  2165  2146  1920  1792  1982  1848  
4   275  ...   209   177   157   147   133   141   120   107    90    89  

[5 rows x 21 columns]
--- COLUNAS ---
['Município

In [3]:
# --- ETAPA 2: PARSING DE STRINGS E EXTRAÇÃO DE CHAVES GEOGRÁFICAS (SERVIÇOS) ---

# Replicando a arquitetura de normalização espacial para a base do setor terciário.
# O desmembramento da string ('UF-Município') é imperativo para garantir o 
# alinhamento geográfico exato (Master Merge) com as demais variáveis do modelo.

print("Iniciando a extração de features geográficas do setor de serviços...")

# 1. PARSING E EXPANSÃO VETORIAL (String Splitting)
# O parâmetro n=1 atua como uma trava de segurança estrutural, blindando o 
# particionamento contra municípios que possuem hífen na própria toponímia.
colunas_separadas = df_emprego_servicos['Município'].str.split('-', n=1, expand=True)

# 2. INSTANCIAÇÃO DAS NOVAS DIMENSÕES ESPACIAIS
# Padronização da nomenclatura para viabilizar o cruzamento futuro em painel.
df_emprego_servicos['Sigla_UF'] = colunas_separadas[0]
df_emprego_servicos['Nome_Municipio'] = colunas_separadas[1]

# 3. OTIMIZAÇÃO DE MEMÓRIA (Drop)
# Exclusão da variável aglutinada original para liberar memória RAM e 
# mitigar redundância informacional no dataset final.
df_emprego_servicos = df_emprego_servicos.drop(columns=['Município'])

# 4. AUDITORIA DA NORMALIZAÇÃO
print("--- Diagnóstico: Extração de Features Geográficas ---")
print("Validação da integridade das chaves 'Sigla_UF' e 'Nome_Municipio':")
print(df_emprego_servicos.head())

--- Tabela com colunas separadas (UF e Município) ---
Veja as duas novas colunas no final:
   2021  2020  2019  2018  2017  2016  2015  2014  2013  2012  ...  2009  \
0   354   328   374   361   367   434   316   324   301   269  ...   210   
1  4668  4490  4730  4725  4498  4329  4547  4359  3982  3708  ...  3000   
2    46    46    51    51    56    49    39    33    35    30  ...    18   
3  5815  5484  5626  5352  5196  4929  4988  4614  4307  3902  ...  2873   
4   536   463   451   384   351   351   345   346   275   233  ...   157   

   2008  2007  2006  2005  2004  2003  2002  Sigla_UF         Nome_Municipio  
0   169   147   171   134   122   100   111        RO  ALTA FLORESTA D OESTE  
1  2754  2525  2264  2127  1939  1720  1460        RO              ARIQUEMES  
2    13    13    12    12    10    10     8        RO                 CABIXI  
3  2383  2165  2146  1920  1792  1982  1848        RO                 CACOAL  
4   147   133   141   120   107    90    89        RO    

In [6]:
# --- ETAPA 3: REESTRUTURAÇÃO PARA PAINEL DE DADOS SETORIAL (MELT - SERVIÇOS) ---

# Para viabilizar a análise longitudinal do Efeito Multiplicador (Spillover), 
# converto a matriz do setor terciário do formato horizontal (Wide) para o vertical (Long).
# Este procedimento padroniza a série histórica para a modelagem econométrica.

print("Iniciando a transposição da série histórica de serviços...")

# 1. DEFINIÇÃO DAS CHAVES PRIMÁRIAS (ID_VARS)
# Seleciono as dimensões geográficas previamente higienizadas e isoladas.
chaves_geograficas = ['Sigla_UF', 'Nome_Municipio']

# 2. OPERAÇÃO DE TRANSPOSIÇÃO (WIDE-TO-LONG)
# Consolido as colunas anuais na dimensão temporal 'Ano' e os quantitativos 
# de vínculos na variável de impacto 'Empregos_servicos'.
df_emprego_servicos_final = df_emprego_servicos.melt(
    id_vars=chaves_geograficas,
    var_name='Ano',
    value_name='Empregos_servicos'
)

# 3. AUDITORIA DE ESTRUTURA E CRONOLOGIA
# Avalio a integridade do empilhamento inspecionando as bordas temporais do painel.
print("--- Diagnóstico: Painel de Empregos em Serviços (Long Format) ---")
print("Cabeçalho (Início da Série Histórica):")
print(df_emprego_servicos_final.head())

print("\nRodapé (Período Mais Antigo/Fim do Empilhamento):")
print(df_emprego_servicos_final.tail())

--- Tabela 'Derretida' (Formato Longo) ---
  Sigla_UF         Nome_Municipio   Ano  Empregos_servicos
0       RO  ALTA FLORESTA D OESTE  2021                354
1       RO              ARIQUEMES  2021               4668
2       RO                 CABIXI  2021                 46
3       RO                 CACOAL  2021               5815
4       RO             CEREJEIRAS  2021                536

--- Últimas linhas (para ver os anos mais antigos) ---
       Sigla_UF Nome_Municipio   Ano  Empregos_servicos
113155       GO       VILA BOA  2002                  7
113156       GO  VILA PROPICIO  2002                  0
113157       GO       IGNORADO  2002                  0
113158       DF       BRASILIA  2002             266130
113159       DF       IGNORADO  2002                  0


In [7]:
# --- ETAPA 3.5: ORDENAÇÃO E INDEXAÇÃO ESPAÇO-TEMPORAL (SERVIÇOS) ---

# A ordenação transversal (Município) e longitudinal (Ano) é o pré-requisito 
# topológico para a estimação de modelos em painel. Softwares econométricos 
# exigem essa hierarquia estrita para calcular o Efeito Multiplicador corretamente.

print("Iniciando o alinhamento longitudinal da série de Serviços...")

# 1. ORDENAÇÃO MULTIDIMENSIONAL (Panel Sorting)
# O algoritmo agrupa as séries por município e ordena cronologicamente o dataset.
df_emprego_servicos_ordenado = df_emprego_servicos_final.sort_values(
    by=['Nome_Municipio', 'Ano']
)

# 2. AUDITORIA DA ORDENAÇÃO DO PAINEL
print("--- Diagnóstico: Matriz do Setor Terciário Ordenada ---")

# [ALERTA METODOLÓGICO]: Como a série legada (1996-2001) não foi integrada 
# neste módulo específico, o período de largada atestado será 2002.
# Atenção ao desbalanceamento de painel na fase de Master Merge!
print("Cabeçalho (Expectativa: Início da ordem alfabética municipal, ano-base 2002):")
print(df_emprego_servicos_ordenado.head())

print("\nRodapé (Expectativa: Fim da ordem alfabética municipal, ano-limite):")
print(df_emprego_servicos_ordenado.tail())

--- Tabela Final ORDENADA ---
Primeiras linhas (deve mostrar o primeiro município e o primeiro ano, 2002):
       Sigla_UF   Nome_Municipio   Ano  Empregos_servicos
112448      RS    PINTO BANDEIRA  2002                  0
106790      RS    PINTO BANDEIRA  2003                  0
101132      RS    PINTO BANDEIRA  2004                  0
95474       RS    PINTO BANDEIRA  2005                  0
89816       RS    PINTO BANDEIRA  2006                  0

Últimas linhas (ordenadas):
      Sigla_UF Nome_Municipio   Ano  Empregos_servicos
27257       SC         ZORTEA  2017                 25
21599       SC         ZORTEA  2018                 25
15941       SC         ZORTEA  2019                 23
10283       SC         ZORTEA  2020                 20
4625        SC         ZORTEA  2021                 14


In [8]:
# --- ETAPA 4: SERIALIZAÇÃO E PERSISTÊNCIA DO PAINEL SETORIAL (OUTPUT DE SERVIÇOS) ---

# Concluo o processamento da variável de Efeito Multiplicador (Emprego em Serviços)
# exportando a matriz ordenada. Este arquivo será cruzado com a base de mineração
# para testar o transbordamento (spillover) econômico no modelo de Controle Sintético.

# 1. DEFINIÇÃO DO REPOSITÓRIO DE SAÍDA (REFINED ZONE)
# Padronizo o roteamento para o diretório segregado ('FINALIZADOS'), mantendo 
# a governança de dados e separando os artefatos brutos dos consolidados.
caminho_saida_csv = r'C:\Users\fabio\TCC\FINALIZADOS\09_Emprego_serv_FINAL.csv'

# 2. EXPORTAÇÃO E INTEGRIDADE ESTRUTURAL
# Invoco o dataframe protegido (df_emprego_servicos_ordenado) com index=False 
# para suprimir índices nativos e garantir a portabilidade limpa para o Stata ou R.
df_emprego_servicos_ordenado.to_csv(
    caminho_saida_csv,
    index=False
)

print(f"--- PROTOCOLO DE EXPORTAÇÃO CONCLUÍDO ---")
print(f"Painel Setorial (Emprego no Setor de Serviços) salvo com sucesso em:")
print(caminho_saida_csv)

--- SUCESSO! ---
Seu arquivo de emprego na mineração consolidado foi salvo em:
C:\Users\fabio\TCC\09_Emprego_serv_FINAL.csv
